In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
# load .mat data
import scipy.io
from scipy.linalg import pinvh

import xarray as xr
# add ../ to the path
import sys
sys.path.append('../')

import src.model as model
from src.utilities import reverse_standardize

%load_ext autoreload
%autoreload 2

data_path       = os.getenv('RAW_DATA_PATH')
param_path      = os.getenv('PARAM_PATH')
output_path     = os.getenv('INFERENCE_READY_DATA_PATH')
posterior_path   = os.getenv('POSTERIOR_PATH')
folder = data_path
# ------------------------------------------------- LOAD SIMULATION DATA -------------------------------------------------
# get the DATA_PATH from the environment variable
XY_train_filename    = "train_data.npz"
XY_valid_filename    = "validation_data.npz"
XY_test_filename     = "test_data.npz"
H_filename           = 'H_gridded.mat'
Y_obs_filename       = "Ns_obs_gridded.mat"
mask_filename        = 'training_mask.mat' 
model_bound_filename = 'training_mask_domain_continuous.mat'
# also the mean and std of Eb for reverse standardization
Eb_mean_filename     = 'trainingAll_Eb_sim_mean.mat'
Eb_std_filename      = 'trainingAll_Eb_sim_std.mat'
# also the mean and std of Ns for standardization of observed data
Ns_mean_filename     = 'trainingAll_Ns_sim_mean_masked.mat'
Ns_std_filename      = 'trainingAll_Ns_sim_std_masked.mat'
# others
coord_filename       = "trainingAll_image_coord.mat"

# preprocessing parameter file: read the csv file
preprocess_param = pd.read_csv(param_path, index_col=0)
Eb_standardize_epsilon = preprocess_param['Eb_standardization_epsilon'].values[0]
Ns_standardize_epsilon = preprocess_param['Ns_standardization_epsilon'].values[0]
print("Eb_standardize_epsilon:", Eb_standardize_epsilon)
print("Ns_standardize_epsilon:", Ns_standardize_epsilon)


flight_mask_data = scipy.io.loadmat(folder + mask_filename) # this is the flightline mask
H_data           = scipy.io.loadmat(folder + H_filename)
model_bound_data = scipy.io.loadmat(folder + model_bound_filename) # this is model domain mask
coord_data       = scipy.io.loadmat(folder + coord_filename)
# standardization data for Eb
Eb_mean          = scipy.io.loadmat(folder + Eb_mean_filename)
Eb_std           = scipy.io.loadmat(folder + Eb_std_filename)

# load
Eb_standardized_train = np.load(output_path + XY_train_filename)['Eb_train']
Ns_standardized_train = np.load(output_path + XY_train_filename)['Ns_train']
Eb_standardized_valid = np.load(output_path + XY_valid_filename)['Eb_validation']
Ns_standardized_valid = np.load(output_path + XY_valid_filename)['Ns_validation']
Eb_standardized_test  = np.load(output_path + XY_test_filename)['Eb_test']
Ns_standardized_test  = np.load(output_path + XY_test_filename)['Ns_test']
# replace nan by 0
Eb_mean = np.nan_to_num(Eb_mean['Eb_mean'], nan=0.0).flatten()
Eb_std  = np.nan_to_num(Eb_std['Eb_std'], nan=0.0).flatten()

# load model boundary
model_bound_data_ori = model_bound_data['in_domain_mask'].astype(bool)

x_coord = coord_data['training_coord'][0][0][0]
y_coord = coord_data['training_coord'][0][0][1]
extent = np.array([x_coord.min(), x_coord.max(), y_coord.max(), y_coord.min()])/1e3 # convert to km (left, right, bottom, top), imshow by default is inverted in y-axis

# -------------------------------------------------- LOAD OBSERVATION DATA --------------------------------------------------
Y_obs_data   = scipy.io.loadmat(folder + Y_obs_filename)
# standardization data for Ns
Ns_mean_data = scipy.io.loadmat(folder + Ns_mean_filename)
Ns_std_data  = scipy.io.loadmat(folder + Ns_std_filename)
Ns_mean = Ns_mean_data['Ns_mean_masked']
Ns_std = Ns_std_data['Ns_std_masked']  

Y_obs_data     = Y_obs_data['atten_rate_grid']
atten_rate_avg = Y_obs_data['atten_avg'][0][0]

# apply flightline masking to the standardization mean and std
flight_mask = flight_mask_data['mask']
flight_mask_bool = flight_mask.astype(bool)
Ns_mean_masked = Ns_mean.copy()
Ns_mean_masked[~flight_mask_bool] = 0 # set the mean to 0
Ns_std_masked = Ns_std.copy()
Ns_std_masked[~flight_mask_bool] = 1 # set the std to 1

Ns_mean_masked = Ns_mean_masked.flatten()   
Ns_std_masked  = Ns_std_masked.flatten()

print('Shape of Eb_standardized_train:', Eb_standardized_train.shape)
print('Shape of Ns_standardized_train:', Ns_standardized_train.shape)
print('Shape of domain mask:', model_bound_data_ori.shape)
print('Shape of flight mask:', flight_mask_bool.shape)

# initialize the model object
md = model.model(extent=extent, coord=(x_coord, y_coord))

# load training data
# first combine all so that our PCA is operated on the whole dataset
Eb_standardized_all = np.concatenate([Eb_standardized_train, 
                                      Eb_standardized_valid, 
                                      Eb_standardized_test], 
                                      axis=3)
Ns_standardized_all = np.concatenate([Ns_standardized_train, 
                                      Ns_standardized_valid, 
                                      Ns_standardized_test], 
                                      axis=3)
md.load_sim_data(Eb_standardized_all, Ns_standardized_all, 
                 domain_mask=model_bound_data_ori, 
                 flight_mask=flight_mask_bool, 
                 show_plot=True,
                 x_name = '$E_b$',
                 y_name = '$N_a$')
plt.savefig("../figs/XY_samples.png",dpi=300, bbox_inches='tight')

md.load_standardization_data(X_mean=Eb_mean, X_std=Eb_std,
                             Y_mean=Ns_mean_masked, Y_std=Ns_std_masked,
                             X_epsilon=Eb_standardize_epsilon, Y_epsilon=Ns_standardize_epsilon)

In [ ]:
from src.validation import bayesian_information_criterion

# reduce dim with PCA. 
md.find_reduction_model_pca(n_component_x=60, n_component_y=15)

# load specifically the train, validation, and test (pre-split) data and apply the fitted PCA
md.load_split_data(X_train=Eb_standardized_train, Y_train=Ns_standardized_train,
                   X_validation=Eb_standardized_valid, Y_validation=Ns_standardized_valid,
                   X_test=Eb_standardized_test, Y_test=Ns_standardized_test)

# based on the BIC sweep plot
n_optimal = 15
md.train_gmm_XY(n_components=n_optimal)
md.plot_gmm_samples_pca(n_samples=5)

In [ ]:
md.load_obs_data(atten_rate_avg)

lambda1, lambda2 = 1,1 # equal mixing
nominal_obs_error = 8 # std of obs error: 8 dB/km

sigma_obs = nominal_obs_error / np.nanmean(md.Y_std)
md.derive_prior(sigma_obs=sigma_obs, lambda1=lambda1, lambda2=lambda2)

# export the calibrated radar attenuation rate to a .npz file
# first bring back to the physical unit

atten_rate_calibrated_standardized = md.atten_rate_calibrated
atten_rate_calibrated = reverse_standardize(atten_rate_calibrated_standardized, 
                                            md.Y_mean, 
                                            md.Y_std,
                                            method='relaxation', 
                                            epsilon=md.Y_epsilon)

np.savez("../data/final-obs-evidence/atten_rate_calibrated.npz",
         atten_rate_calibrated=atten_rate_calibrated,
         flight_mask=md.flight_mask,
         x = md.coord[0],
         y = md.coord[1])




### Load the observations

In [ ]:
from src.ice import compute_pmp

# load the thawed base and frozen base data
frozen_base_filename = folder + "../../basal-frozen-mask/frozen_mask.mat"
thawed_base_filename = folder + "../../basal-water-mask/water_mask.mat"
frozen_mask = scipy.io.loadmat(frozen_base_filename)['frozen_mask']
thawed_mask = scipy.io.loadmat(thawed_base_filename)['water_mask']
frozen_mask_data = frozen_mask['data'][0][0]
thawed_mask_data = thawed_mask['data'][0][0]

Eb_mean_data = scipy.io.loadmat(folder + Eb_mean_filename)
Eb_std_data = scipy.io.loadmat(folder + Eb_std_filename)
# replace nan by 0
Eb_mean_data = np.nan_to_num(Eb_mean_data['Eb_mean'], nan=0.0)
Eb_std_data = np.nan_to_num(Eb_std_data['Eb_std'], nan=0.0)

# compute pmp
H = H_data['H_struct']['H'][0][0].flatten()
X = H_data['H_struct']['X'][0][0].flatten()
Y = H_data['H_struct']['Y'][0][0].flatten()
pmp = compute_pmp(H)

# into a vector
Eb_mean_data = Eb_mean_data.flatten()
Eb_std_data = Eb_std_data.flatten()
frozen_mask_data = frozen_mask_data.flatten()
thawed_mask_data = thawed_mask_data.flatten()

# set fractional area to 1 for all locations (assumption for now, can be relaxed later)
thawed_fractional_area = np.zeros_like(thawed_mask_data)
frozen_fractional_area = np.zeros_like(frozen_mask_data)
thawed_fractional_area[thawed_mask_data == 1] = 1.0
frozen_fractional_area[frozen_mask_data == 1] = 1.0

# load
md.load_evidence(thawed_mask_data, thawed_fractional_area,
                 frozen_mask_data, frozen_fractional_area,
                 pmp,
                 show_plot=True)

# Parameteric studies of the discrepancy modeling

In [ ]:
from itertools import product
from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd
import scipy as sp

from src.lowRankGP import lowRankGP
from src.gmmAugmented import gmmAugmented
from src.latentEnthalpyModel import (
    latentEnthalpyModel,
    BasalEvidence,
    PosteriorConfig,
)
from src.enthalpyPosterior import enthalpyPosterior
from src.posteriorObjective import posteriorObjective


# ------------------------------------------------------------
# Fixed data and posterior configuration
# ------------------------------------------------------------
xx = md.coord[1].ravel()
yy = md.coord[0].ravel()

hx = float(xx[1] - xx[0])
hy = float(yy[1] - yy[0])

evidence = BasalEvidence.from_arrays(
    Tpmp=md.pmp,
    thawed_mask=thawed_mask_data,
    frozen_mask=frozen_mask_data,
    dw=thawed_fractional_area,
    df=frozen_fractional_area,
)

config = PosteriorConfig(
    beta=0.5,
    beta_w=0.005,
)

n_gmm_dim = len(md.x_indices)

output_dir = Path("matern_sigma_sweep")
output_dir.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# Experiment grid: 45 grid points + original baseline
# ------------------------------------------------------------
sigma_values = np.geomspace(0.01, 10.0, 5)
nu_values = [0.1, 0.5, 1.5]
rho_values = [0.5, 1.0, 2.0]

experiments = [
    {
        "sigma_amp": 2.0,
        "nu": 0.1,
        "rho": 1.0,
        "is_baseline": True,
    }
]

experiments.extend(
    {
        "sigma_amp": float(sigma),
        "nu": float(nu),
        "rho": float(rho),
        "is_baseline": False,
    }
    for sigma, nu, rho in product(
        sigma_values, nu_values, rho_values
    )
)

print(f"Running {len(experiments)} experiments")


# ------------------------------------------------------------
# Run sweep
# ------------------------------------------------------------
records = []

for experiment_id, params in enumerate(experiments):
    start_time = perf_counter()

    record = {
        "experiment_id": experiment_id,
        **params,
        "success": False,
        "status": np.nan,
        "message": "",
        "initial_objective": np.nan,
        "final_objective": np.nan,
        "objective_reduction": np.nan,
        "gradient_inf_norm": np.nan,
        "nit": np.nan,
        "nfev": np.nan,
        "njev": np.nan,
        "theta_file": "",
    }

    print(
        f"[{experiment_id + 1:02d}/{len(experiments)}] "
        f"sigma_amp={params['sigma_amp']:.5g}, "
        f"nu={params['nu']:.3g}, "
        f"rho={params['rho']:.3g}"
    )

    try:
        gp_prior = lowRankGP(
            domain_mask=md.domain_mask,
            hx=hx,
            hy=hy,
            n_mode=100,
            pca_basis=md.pca_x.components_.T,
            sigma_amp=params["sigma_amp"],
            nu=params["nu"],
            rho=params["rho"],
        )

        # Recreate this as well to avoid carrying mutable state
        # between experiments.
        sim_prior = gmmAugmented(
            gmm=md.gmm_prop,
        )

        forward_model = latentEnthalpyModel(
            pca=md.pca_x,
            gp=gp_prior,
            Eb_mean=md.X_mean,
            Eb_std=md.X_std,
            Eb_epsilon=md.X_epsilon,
            method="relaxation",
            Tpmp=md.pmp,
        )

        posterior = enthalpyPosterior(
            forward_model=forward_model,
            evidence=evidence,
            gmm=sim_prior,
            gp=gp_prior,
            config=config,
        )

        n_gp_dim = gp_prior.n_mode

        objective = posteriorObjective(
            posterior=posterior,
            gp_dimension=n_gp_dim,
            gmm_dimension=n_gmm_dim,
        )

        # Same initialization for every experiment.
        # NumPy is the native input format for scipy.optimize.
        theta0 = np.zeros(
            n_gp_dim + n_gmm_dim,
            dtype=np.float64,
        )

        initial_objective = float(objective.value(theta0))
        record["initial_objective"] = initial_objective

        if not np.isfinite(initial_objective):
            raise ValueError("Initial objective is not finite.")

        result = sp.optimize.minimize(
            fun=objective.value,
            x0=theta0,
            jac=objective.gradient,
            method="L-BFGS-B",
            options={
                "maxiter": 500,
                "maxfun": 10000,
                "ftol": 1e-9,
                "gtol": 1e-6,
                "maxls": 30,
            },
        )

        theta_file = output_dir / f"theta_{experiment_id:03d}.npz"

        # Save even unsuccessful runs for diagnosis or restarting.
        np.savez_compressed(
            theta_file,
            theta=result.x,
            theta_gp=result.x[:n_gp_dim],
            theta_gmm=result.x[n_gp_dim:],
            sigma_amp=params["sigma_amp"],
            nu=params["nu"],
            rho=params["rho"],
        )

        final_objective = float(result.fun)
        gradient = np.asarray(result.jac, dtype=float)

        record.update(
            success=bool(result.success),
            status=int(result.status),
            message=str(result.message),
            final_objective=final_objective,
            objective_reduction=initial_objective - final_objective,
            gradient_inf_norm=float(
                np.linalg.norm(gradient, ord=np.inf)
            ),
            nit=int(result.nit),
            nfev=int(result.nfev),
            njev=int(getattr(result, "njev", 0)),
            theta_file=str(theta_file),
        )

        print(
            f"    success={result.success}, "
            f"objective={final_objective:.6g}, "
            f"iterations={result.nit}"
        )

    except Exception as exc:
        record["message"] = f"{type(exc).__name__}: {exc}"
        print(f"    FAILED: {record['message']}")

    record["elapsed_seconds"] = perf_counter() - start_time
    records.append(record)

    # Checkpoint after each experiment.
    pd.DataFrame(records).to_csv(
        output_dir / "sweep_results.csv",
        index=False,
    )


# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------
results_df = pd.DataFrame(records)

converged = results_df.loc[
    results_df["success"]
    & np.isfinite(results_df["final_objective"])
].copy()

# This is an objective-based diagnostic ranking, not necessarily
# a statistically valid ranking of hyperparameter quality.
objective_ranking = converged.sort_values("final_objective")

summary_columns = [
    "experiment_id",
    "is_baseline",
    "sigma_amp",
    "nu",
    "rho",
    "final_objective",
    "gradient_inf_norm",
    "nit",
    "elapsed_seconds",
]

print(
    f"\nConverged: {len(converged)}/{len(results_df)}"
)

if not objective_ranking.empty:
    print("\nLowest-objective converged runs:")
    print(
        objective_ranking[summary_columns]
        .head(10)
        .to_string(index=False)
    )

    objective_ranking.to_csv(
        output_dir / "converged_objective_ranking.csv",
        index=False,
    )
else:
    print("\nNo converged runs; inspect sweep_results.csv.")


NameError: name 'md' is not defined

# Plotting the results

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

results_dir = Path("matern_sigma_sweep")
runs = pd.read_csv(results_dir / "sweep_results.csv")
runs = runs.sort_values(["nu", "rho", "sigma_amp"])

ncols = 5
nrows = int(np.ceil(len(runs) / ncols))

fig, axes = plt.subplots(
    nrows, ncols,
    figsize=(20, 3.5 * nrows),
    squeeze=False,
    layout="constrained",
)

image = None

for ax, (_, run) in zip(axes.flat, runs.iterrows()):
    ax.set_title(
        f"Run {int(run.experiment_id)}\n"
        f"σ={run.sigma_amp:.3g}, ν={run.nu:g}, ρ={run.rho:g}"
    )

    theta_file = (
        results_dir / f"theta_{int(run.experiment_id):03d}.npz"
    )

    if not theta_file.exists():
        ax.text(
            0.5, 0.5, "No saved result",
            ha="center", va="center",
            transform=ax.transAxes,
        )
        ax.set_axis_off()
        continue

    with np.load(theta_file) as saved:
        map_z_gp = saved["theta_gp"].copy()
        map_z_gmm = saved["theta_gmm"].copy()

    physical_result_MAP = forward_model.physical_state(
        map_z_gp, map_z_gmm
    )

    result = (
        physical_result_MAP["delta_T"]
        .reshape(md.nx, md.ny)
        .detach()
        .cpu()
        .numpy()
        .copy()
    )
    result = np.where(md.domain_mask, result, np.nan)

    image = ax.imshow(
        result,
        cmap="hot",
        vmin=0,
        vmax=10,
    )
    ax.invert_yaxis()
    ax.set_xticks([])
    ax.set_yticks([])

for ax in list(axes.flat)[len(runs):]:
    ax.set_visible(False)

if image is not None:
    fig.colorbar(
        image,
        ax=[ax for ax in axes.flat if ax.get_visible()],
        label="ΔT",
        shrink=0.7,
    )

fig.savefig(
    results_dir / "delta_T_grid.png",
    dpi=200,
    bbox_inches="tight",
)
plt.show()


#### Plotting the pairwise comparison

In [ ]:
from itertools import product
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm


# ------------------------------------------------------------
# Load sweep results; exclude the extra baseline
# ------------------------------------------------------------
results_dir = Path("matern_sigma_sweep")
runs = pd.read_csv(results_dir / "sweep_results.csv")

if "is_baseline" in runs.columns:
    is_baseline = (
        runs["is_baseline"].astype(str).str.lower().isin(["true", "1"])
    )
    runs = runs.loc[~is_baseline].copy()

parameters = ["sigma_amp", "nu", "rho"]
labels = {"sigma_amp": "σ", "nu": "ν", "rho": "ρ"}

levels = {
    parameter: (
        float(runs[parameter].min()),
        float(runs[parameter].max()),
    )
    for parameter in parameters
}


# ------------------------------------------------------------
# Load physical fields only once per run
# ------------------------------------------------------------
field_cache = {}


def load_field(settings):
    mask = np.ones(len(runs), dtype=bool)

    for parameter, value in settings.items():
        mask &= np.isclose(runs[parameter].to_numpy(), value)

    matches = runs.loc[mask]

    if len(matches) != 1:
        raise ValueError(
            f"Expected one run for {settings}; found {len(matches)}."
        )

    run_id = int(matches.iloc[0]["experiment_id"])

    if run_id not in field_cache:
        theta_file = results_dir / f"theta_{run_id:03d}.npz"

        with np.load(theta_file) as saved:
            map_z_gp = saved["theta_gp"].copy()
            map_z_gmm = saved["theta_gmm"].copy()

        physical_result_MAP = forward_model.physical_state(
            map_z_gp, map_z_gmm
        )

        field = (
            physical_result_MAP["delta_T"]
            .reshape(md.nx, md.ny)
            .detach()
            .cpu()
            .numpy()
            .copy()
        )

        field_cache[run_id] = np.where(
            md.domain_mask, field, np.nan
        )

    return field_cache[run_id]


# ------------------------------------------------------------
# Compute all 12 difference maps
# ------------------------------------------------------------
comparisons = {}

for parameter in parameters:
    other_parameters = [
        p for p in parameters if p != parameter
    ]
    low, high = levels[parameter]

    panels = []

    for reference_values in product(
        *(levels[p] for p in other_parameters)
    ):
        reference = dict(zip(other_parameters, reference_values))

        field_low = load_field({**reference, parameter: low})
        field_high = load_field({**reference, parameter: high})

        panels.append({
            "reference": reference,
            "difference": field_high - field_low,
        })

    comparisons[parameter] = panels


# Shared symmetric limits across all three figures.
finite_values = [
    panel["difference"][np.isfinite(panel["difference"])]
    for panels in comparisons.values()
    for panel in panels
]

finite_values = [values for values in finite_values if values.size]

if not finite_values:
    raise ValueError("No finite difference values found.")

limit = max(np.max(np.abs(values)) for values in finite_values)
limit = float(limit) if limit > 0 else 1.0

# To use a fixed difference scale instead:
limit = 10.0

norm = TwoSlopeNorm(vmin=-limit, vcenter=0, vmax=limit)


# ------------------------------------------------------------
# Plot one 2×2 figure per compared parameter
# ------------------------------------------------------------
for parameter, panels in comparisons.items():
    low, high = levels[parameter]
    other_parameters = [
        p for p in parameters if p != parameter
    ]

    fig, axes = plt.subplots(
        2, 2,
        figsize=(11, 9),
        layout="constrained",
    )

    for ax, panel in zip(axes.flat, panels):
        image = ax.imshow(
            panel["difference"],
            cmap="RdBu_r",
            norm=norm,
        )

        ax.invert_yaxis()
        ax.set_xticks([])
        ax.set_yticks([])

        ax.set_title(
            ", ".join(
                f"{labels[p]}={panel['reference'][p]:.3g}"
                for p in other_parameters
            )
        )

    fig.suptitle(
        f"ΔT difference: {labels[parameter]}={high:g} "
        f"minus {labels[parameter]}={low:g}\n"
        f"Rows: low/high {labels[other_parameters[0]]}; "
        f"columns: low/high {labels[other_parameters[1]]}",
        fontsize=14,
    )

    fig.colorbar(
        image,
        ax=axes.ravel().tolist(),
        label="ΔT(high) − ΔT(low)",
        shrink=0.85,
    )

    fig.savefig(
        results_dir / f"delta_T_difference_{parameter}.png",
        dpi=200,
        bbox_inches="tight",
    )

plt.show()